# Debug Notebook: GPU Optimization Benchmark

This notebook benchmarks speculative decoding with GPU optimization switches OFF vs ON,
then produces an automatic delta table for:
- draft time (`draft_elapsed_s`)
- verify time (`verify_elapsed_s`)
- end-to-end latency (`latency_s`)

Outputs are saved to `results/gpu_opt_benchmark_runs.csv` and `results/gpu_opt_benchmark_delta.csv`.

In [1]:
import hashlib
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'torch is required for this notebook. Select a Python environment with PyTorch installed.'
    ) from exc

# Resolve project root robustly (works from notebook root or subfolders).
search_starts = []
if '__file__' in globals():
    search_starts.append(Path(__file__).resolve().parent)
search_starts.append(Path.cwd().resolve())

project_root = None
seen = set()
for start in search_starts:
    for p in [start, *start.parents]:
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'speculative.py').exists():
            project_root = p
            break
    if project_root is not None:
        break

if project_root is None:
    raise RuntimeError('Could not find project root containing src/speculative.py')

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Offline-first behavior for reproducibility in debug sessions.
os.environ.setdefault('SPECDEC_HF_OFFLINE_FIRST', '1')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for this benchmark notebook.')

import speculative as spec_mod
from config import DRAFT_MODELS, DRAFT_QUANT, TARGET_MODEL_ID, TARGET_QUANT, REGIMES
from speculative import load_model_on_device
from utils import set_seed

results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
manifests_dir = project_root / 'manifests'

print(f'Project root: {project_root}')
print(f'CUDA device count: {torch.cuda.device_count()}')

Project root: C:\Working\speculative-decoding-main_v8\speculative-decoding-main
CUDA device count: 2


In [2]:
# Benchmark controls
DEVICE = 'cuda:0'
DRAFT_LABEL = '0.5B'
REGIME_NAME = 'deterministic'   # set to 'stochastic' if needed
K = 8
MAX_NEW_TOKENS = 64
N_PER_TASK = 4
BASE_SEED = 2026
WARMUP_NEW_TOKENS = 16

# Mode switch:
# - 'paired_on_off': only OFF vs ON (default)
# - 'single_flag_ablation': baseline OFF + one-flag-at-a-time + optional ALL_ON
BENCHMARK_MODE = 'paired_on_off'
INCLUDE_ALL_ON_IN_ABLATION = True

FLAG_NAMES = [
    'GPU_USE_SEPARATE_STREAMS',
    'GPU_PREALLOCATE_STEP_BUFFERS',
    'GPU_USE_STABLE_STEP_SHAPES',
    'GPU_TRY_CUDA_GRAPHS',
]

BASELINE_FLAGS = {name: False for name in FLAG_NAMES}
ALL_ON_FLAGS = {
    'GPU_USE_SEPARATE_STREAMS': True,
    'GPU_PREALLOCATE_STEP_BUFFERS': True,
    'GPU_USE_STABLE_STEP_SHAPES': True,
    'GPU_TRY_CUDA_GRAPHS': False,
}

def build_conditions(mode: str) -> list[dict]:
    base = {'condition': 'gpu_opt_off', **BASELINE_FLAGS}
    all_on = {'condition': 'gpu_opt_on', **ALL_ON_FLAGS}

    if mode == 'paired_on_off':
        return [base, all_on]

    if mode == 'single_flag_ablation':
        rows = [base]
        for flag in FLAG_NAMES:
            cfg = {'condition': f'only_{flag.lower()}', **BASELINE_FLAGS}
            cfg[flag] = True
            rows.append(cfg)
        if INCLUDE_ALL_ON_IN_ABLATION:
            rows.append(all_on)
        return rows

    raise ValueError(f'Unknown BENCHMARK_MODE: {mode}')

CONDITIONS = build_conditions(BENCHMARK_MODE)

print('Benchmark config:')
print(f'  device={DEVICE}, target_quant={TARGET_QUANT}, draft_quant={DRAFT_QUANT}')
print(f'  draft={DRAFT_LABEL}, regime={REGIME_NAME}, k={K}, max_new_tokens={MAX_NEW_TOKENS}')
print(f'  n_per_task={N_PER_TASK}, warmup_new_tokens={WARMUP_NEW_TOKENS}')
print(f'  benchmark_mode={BENCHMARK_MODE}, conditions={len(CONDITIONS)}')
pd.DataFrame(CONDITIONS)[['condition', *FLAG_NAMES]]

Benchmark config:
  device=cuda:0, target_quant=fp16, draft_quant=fp16
  draft=0.5B, regime=deterministic, k=8, max_new_tokens=64
  n_per_task=4, warmup_new_tokens=16
  benchmark_mode=paired_on_off, conditions=2


,condition,GPU_USE_SEPARATE_STREAMS,GPU_PREALLOCATE_STEP_BUFFERS,GPU_USE_STABLE_STEP_SHAPES,GPU_TRY_CUDA_GRAPHS
0,gpu_opt_off,False,False,False,False
1,gpu_opt_on,True,True,True,False


In [3]:
def load_prompt_rows(n_per_task: int = 4) -> list[dict]:
    rows = []
    for task in ['gsm8k', 'mmlu', 'cnndm']:
        path = manifests_dir / f'{task}_data.json'
        if not path.exists():
            continue
        with open(path, 'r') as f:
            data = json.load(f)
        for item in data[:n_per_task]:
            prompt = item.get('prompt')
            if prompt:
                rows.append({
                    'task': task,
                    'sample_id': item.get('sample_id', ''),
                    'prompt': prompt,
                })
    if not rows:
        raise RuntimeError('No prompts found in manifests/*_data.json')
    return rows

prompt_rows = load_prompt_rows(N_PER_TASK)
print(f'Loaded prompts: {len(prompt_rows)}')
pd.DataFrame(prompt_rows)[['task', 'sample_id']].head(12)

Loaded prompts: 12


,task,sample_id
0,gsm8k,gsm8k_2
1,gsm8k,gsm8k_11
2,gsm8k,gsm8k_15
3,gsm8k,gsm8k_21
4,mmlu,mmlu_abstract_algebra_0
5,mmlu,mmlu_abstract_algebra_1
6,mmlu,mmlu_abstract_algebra_2
7,mmlu,mmlu_abstract_algebra_3
8,cnndm,cnndm_3
9,cnndm,cnndm_31


In [4]:
# Load models once and reuse across both conditions.
target_model, target_tokenizer = load_model_on_device(
    TARGET_MODEL_ID,
    device=DEVICE,
    quant_mode=TARGET_QUANT,
)
draft_model, _ = load_model_on_device(
    DRAFT_MODELS[DRAFT_LABEL],
    device=DEVICE,
    quant_mode=DRAFT_QUANT,
)

print('Models loaded.')
print(f'  target: {TARGET_MODEL_ID}')
print(f'  draft : {DRAFT_MODELS[DRAFT_LABEL]}')

Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading model: Qwen/Qwen2.5-0.5B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Models loaded.
  target: Qwen/Qwen2.5-3B-Instruct
  draft : Qwen/Qwen2.5-0.5B-Instruct


In [5]:
def apply_condition_flags(cfg: dict) -> None:
    spec_mod.GPU_USE_SEPARATE_STREAMS = bool(cfg['GPU_USE_SEPARATE_STREAMS'])
    spec_mod.GPU_PREALLOCATE_STEP_BUFFERS = bool(cfg['GPU_PREALLOCATE_STEP_BUFFERS'])
    spec_mod.GPU_USE_STABLE_STEP_SHAPES = bool(cfg['GPU_USE_STABLE_STEP_SHAPES'])
    spec_mod.GPU_TRY_CUDA_GRAPHS = bool(cfg['GPU_TRY_CUDA_GRAPHS'])

def run_condition(condition_cfg: dict, prompts: list[dict], base_seed: int = 0) -> pd.DataFrame:
    condition_name = condition_cfg['condition']
    apply_condition_flags(condition_cfg)
    regime = REGIMES[REGIME_NAME]

    # Warm-up (excluded from metrics).
    warm_prompt = prompts[0]['prompt']
    set_seed(base_seed)
    _ = spec_mod.speculative_decode_sample(
        target_model,
        draft_model,
        target_tokenizer,
        warm_prompt,
        max_new_tokens=min(WARMUP_NEW_TOKENS, MAX_NEW_TOKENS),
        k=K,
        temperature=regime.temperature,
        top_p=regime.top_p,
        return_timing_breakdown=True,
    )

    records = []
    for i, row in enumerate(prompts):
        seed = base_seed + i
        set_seed(seed)
        out = spec_mod.speculative_decode_sample(
            target_model,
            draft_model,
            target_tokenizer,
            row['prompt'],
            max_new_tokens=MAX_NEW_TOKENS,
            k=K,
            temperature=regime.temperature,
            top_p=regime.top_p,
            return_timing_breakdown=True,
        )

        output_text = str(out.get('output_text', ''))
        output_hash = hashlib.sha256(output_text.encode('utf-8')).hexdigest()

        records.append({
            'condition': condition_name,
            'task': row['task'],
            'sample_id': row['sample_id'],
            'seed': seed,
            'k': K,
            'max_new_tokens': MAX_NEW_TOKENS,
            'latency_s': float(out.get('latency_s', 0.0)),
            'draft_elapsed_s': float(out.get('draft_elapsed_s', 0.0)),
            'verify_elapsed_s': float(out.get('verify_elapsed_s', 0.0)),
            'ttft_ms': float(out.get('ttft_ms', 0.0)),
            'num_tokens': int(out.get('num_tokens', 0)),
            'alpha': float(out.get('alpha', 0.0)),
            'B_eff': float(out.get('B_eff', 0.0)),
            'gpu_stream_pipeline': bool(out.get('gpu_stream_pipeline', False)),
            'gpu_stable_step_shapes': bool(out.get('gpu_stable_step_shapes', False)),
            'gpu_graph_capture': out.get('gpu_graph_capture', 'unknown'),
            'output_hash': output_hash,
            'output_text': output_text,
        })

    return pd.DataFrame(records)

In [6]:
all_runs = []
for idx, cfg in enumerate(CONDITIONS):
    print(f"Running {cfg['condition']} ({idx + 1}/{len(CONDITIONS)})...")
    df_cond = run_condition(cfg, prompt_rows, base_seed=BASE_SEED)
    all_runs.append(df_cond)

df_runs = pd.concat(all_runs, ignore_index=True)
print(f'Benchmark rows: {len(df_runs)}')
print('Conditions run:', sorted(df_runs['condition'].unique()))
df_runs.head()

Running gpu_opt_off (1/2)...
Running gpu_opt_on (2/2)...
Benchmark rows: 24
Conditions run: ['gpu_opt_off', 'gpu_opt_on']


,condition,task,sample_id,seed,k,max_new_tokens,latency_s,draft_elapsed_s,verify_elapsed_s,ttft_ms,num_tokens,alpha,B_eff,gpu_stream_pipeline,gpu_stable_step_shapes,gpu_graph_capture,output_hash,output_text
0,gpu_opt_off,gsm8k,gsm8k_2,2026,8,64,2.3087,1.926528,0.377840,145.14,64,0.3582,2.82,False,False,disabled,446da7bbb09b7af938cb2767cdb0ef7b5b965060b0eb42...,Step 1: Calculate the total cost of the house...
1,gpu_opt_off,gsm8k,gsm8k_11,2027,8,64,1.1547,0.948557,0.204478,145.47,64,0.8485,6.22,False,False,disabled,f5ca627d1549c4c983b003f864a9501261e3db008fe228...,Step 1: Calculate the cost of the donuts.\nTo...
2,gpu_opt_off,gsm8k,gsm8k_15,2028,8,64,3.2183,2.672520,0.539297,145.52,64,0.2151,1.67,False,False,disabled,9e7b34d288ae138de6ab0e9a625911ebb2fb189df7be90...,Express your answer to the nearest dollar.\nS...
3,gpu_opt_off,gsm8k,gsm8k_21,2029,8,64,1.8334,1.518369,0.311519,144.07,64,0.4811,3.64,False,False,disabled,5cad46859161a0f816b6192d4d8e4ec41f40c15d47da59...,Let's solve this problem step by step.\n\n1. ...
4,gpu_opt_off,mmlu,mmlu_abstract_algebra_0,2030,8,64,2.1577,1.777679,0.375687,143.96,64,0.3871,2.82,False,False,disabled,b20a5682b9a2b3b1dccfd5646b14a69c685f8ebac9359f...,D. 6\n\nTo determine the degree of the field ...


In [7]:
# Deterministic output-equivalence checks (hash-based).
KEYS = ['task', 'sample_id', 'seed', 'k', 'max_new_tokens']
baseline_condition = 'gpu_opt_off' if 'gpu_opt_off' in set(df_runs['condition']) else sorted(df_runs['condition'].unique())[0]

base_cols = KEYS + ['output_hash', 'output_text', 'alpha', 'B_eff']
base_df = df_runs[df_runs['condition'] == baseline_condition][base_cols].rename(columns={
    'output_hash': 'output_hash_base',
    'output_text': 'output_text_base',
    'alpha': 'alpha_base',
    'B_eff': 'B_eff_base',
})

eq_rows = []
for cond in sorted(df_runs['condition'].unique()):
    if cond == baseline_condition:
        continue
    cmp_df = df_runs[df_runs['condition'] == cond][base_cols].rename(columns={
        'output_hash': 'output_hash_cmp',
        'output_text': 'output_text_cmp',
        'alpha': 'alpha_cmp',
        'B_eff': 'B_eff_cmp',
    })
    merged = base_df.merge(cmp_df, on=KEYS, how='inner')
    if merged.empty:
        continue

    hash_match = merged['output_hash_base'] == merged['output_hash_cmp']
    alpha_match = np.isclose(merged['alpha_base'], merged['alpha_cmp'], atol=1e-8)
    beff_match = np.isclose(merged['B_eff_base'], merged['B_eff_cmp'], atol=1e-8)

    mismatch_ids = merged.loc[~hash_match, 'sample_id'].tolist()[:5]
    eq_rows.append({
        'baseline_condition': baseline_condition,
        'compare_condition': cond,
        'n_pairs': int(len(merged)),
        'n_output_mismatch': int((~hash_match).sum()),
        'output_match_rate_pct': float(hash_match.mean() * 100.0),
        'alpha_match_rate_pct': float(alpha_match.mean() * 100.0),
        'beff_match_rate_pct': float(beff_match.mean() * 100.0),
        'example_mismatch_sample_ids': ';'.join(mismatch_ids),
    })

equivalence_table = pd.DataFrame(eq_rows)
equivalence_table if not equivalence_table.empty else pd.DataFrame([{'note': 'No comparison condition available.'}])

,baseline_condition,compare_condition,n_pairs,n_output_mismatch,output_match_rate_pct,alpha_match_rate_pct,beff_match_rate_pct,example_mismatch_sample_ids
0,gpu_opt_off,gpu_opt_on,12,0,100.0,100.0,100.0,


In [8]:
# Mean delta table (paired per sample vs baseline).
METRICS = ['draft_elapsed_s', 'verify_elapsed_s', 'latency_s', 'ttft_ms']
KEYS = ['task', 'sample_id', 'seed', 'k', 'max_new_tokens']
baseline_condition = 'gpu_opt_off' if 'gpu_opt_off' in set(df_runs['condition']) else sorted(df_runs['condition'].unique())[0]

base_df = df_runs[df_runs['condition'] == baseline_condition][KEYS + METRICS].rename(columns={m: f'{m}_base' for m in METRICS})

paired_deltas = []
for cond in sorted(df_runs['condition'].unique()):
    if cond == baseline_condition:
        continue
    cmp_df = df_runs[df_runs['condition'] == cond][KEYS + METRICS].rename(columns={m: f'{m}_cmp' for m in METRICS})
    merged = base_df.merge(cmp_df, on=KEYS, how='inner')
    if merged.empty:
        continue

    for m in METRICS:
        merged[f'{m}_delta'] = merged[f'{m}_cmp'] - merged[f'{m}_base']
        merged[f'{m}_delta_pct'] = np.where(
            merged[f'{m}_base'] != 0,
            merged[f'{m}_delta'] / merged[f'{m}_base'] * 100.0,
            np.nan,
        )

    merged['baseline_condition'] = baseline_condition
    merged['compare_condition'] = cond
    paired_deltas.append(merged)

if paired_deltas:
    paired_df = pd.concat(paired_deltas, ignore_index=True)
else:
    paired_df = pd.DataFrame()

mean_rows = []
for cond in sorted(paired_df['compare_condition'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_condition'] == cond]
    for m in METRICS:
        base_mean = float(block[f'{m}_base'].mean())
        cmp_mean = float(block[f'{m}_cmp'].mean())
        abs_delta = float(block[f'{m}_delta'].mean())
        pct_delta = (abs_delta / base_mean * 100.0) if base_mean != 0 else np.nan
        mean_rows.append({
            'baseline_condition': baseline_condition,
            'compare_condition': cond,
            'metric': m,
            'baseline_mean': base_mean,
            'compare_mean': cmp_mean,
            'abs_delta_compare_minus_base': abs_delta,
            'pct_delta_compare_minus_base': pct_delta,
            'improvement_pct_lower_is_better': (-pct_delta) if pd.notna(pct_delta) else np.nan,
            'ratio_compare_over_base': (cmp_mean / base_mean) if base_mean != 0 else np.nan,
            'n_pairs': int(len(block)),
        })

mean_delta_table = pd.DataFrame(mean_rows)
mean_delta_table

,baseline_condition,compare_condition,metric,baseline_mean,compare_mean,abs_delta_compare_minus_base,pct_delta_compare_minus_base,improvement_pct_lower_is_better,ratio_compare_over_base,n_pairs
0,gpu_opt_off,gpu_opt_on,draft_elapsed_s,1.865171,1.889037,0.023866,1.279588,-1.279588,1.012796,12
1,gpu_opt_off,gpu_opt_on,verify_elapsed_s,0.402933,0.413529,0.010596,2.629613,-2.629613,1.026296,12
2,gpu_opt_off,gpu_opt_on,latency_s,2.272600,2.308058,0.035458,1.560254,-1.560254,1.015603,12
3,gpu_opt_off,gpu_opt_on,ttft_ms,170.904167,176.667500,5.763333,3.372260,-3.372260,1.033723,12


In [9]:
# Robust delta tables: median and p90 of paired deltas.
robust_rows = []
for cond in sorted(paired_df['compare_condition'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_condition'] == cond]
    for m in METRICS:
        d = block[f'{m}_delta'].to_numpy()
        robust_rows.append({
            'baseline_condition': baseline_condition,
            'compare_condition': cond,
            'metric': m,
            'median_delta_compare_minus_base': float(np.median(d)),
            'p90_delta_compare_minus_base': float(np.quantile(d, 0.90)),
            'p10_delta_compare_minus_base': float(np.quantile(d, 0.10)),
            'median_abs_delta': float(np.median(np.abs(d))),
            'p90_abs_delta': float(np.quantile(np.abs(d), 0.90)),
            'n_pairs': int(len(d)),
        })

robust_delta_table = pd.DataFrame(robust_rows)

print('Median delta table:')
display(robust_delta_table[['baseline_condition','compare_condition','metric','median_delta_compare_minus_base','median_abs_delta','n_pairs']])

print('P90 delta table:')
display(robust_delta_table[['baseline_condition','compare_condition','metric','p90_delta_compare_minus_base','p90_abs_delta','n_pairs']])

Median delta table:


,baseline_condition,compare_condition,metric,median_delta_compare_minus_base,median_abs_delta,n_pairs
0,gpu_opt_off,gpu_opt_on,draft_elapsed_s,0.007549,0.011950,12
1,gpu_opt_off,gpu_opt_on,verify_elapsed_s,0.003891,0.005504,12
2,gpu_opt_off,gpu_opt_on,latency_s,0.011350,0.017100,12
3,gpu_opt_off,gpu_opt_on,ttft_ms,2.235000,2.235000,12


P90 delta table:


,baseline_condition,compare_condition,metric,p90_delta_compare_minus_base,p90_abs_delta,n_pairs
0,gpu_opt_off,gpu_opt_on,draft_elapsed_s,0.072251,0.072251,12
1,gpu_opt_off,gpu_opt_on,verify_elapsed_s,0.026785,0.026785,12
2,gpu_opt_off,gpu_opt_on,latency_s,0.094150,0.094150,12
3,gpu_opt_off,gpu_opt_on,ttft_ms,6.423000,6.423000,12


In [10]:
# Paired bootstrap CI table for mean deltas.
BOOTSTRAP_SAMPLES = 20000
BOOTSTRAP_SEED = 42
rng = np.random.default_rng(BOOTSTRAP_SEED)

ci_rows = []
for cond in sorted(paired_df['compare_condition'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_condition'] == cond]
    n = len(block)
    if n == 0:
        continue

    for m in METRICS:
        d = block[f'{m}_delta'].to_numpy()
        means = np.empty(BOOTSTRAP_SAMPLES, dtype=float)
        for i in range(BOOTSTRAP_SAMPLES):
            idx = rng.integers(0, n, n)
            means[i] = float(d[idx].mean())
        lo, hi = np.quantile(means, [0.025, 0.975])
        ci_rows.append({
            'baseline_condition': baseline_condition,
            'compare_condition': cond,
            'metric': m,
            'mean_delta_compare_minus_base': float(d.mean()),
            'ci95_low': float(lo),
            'ci95_high': float(hi),
            'ci_excludes_zero': bool((lo > 0) or (hi < 0)),
            'n_pairs': int(n),
            'bootstrap_samples': int(BOOTSTRAP_SAMPLES),
        })

bootstrap_ci_table = pd.DataFrame(ci_rows)
bootstrap_ci_table

,baseline_condition,compare_condition,metric,mean_delta_compare_minus_base,ci95_low,ci95_high,ci_excludes_zero,n_pairs,bootstrap_samples
0,gpu_opt_off,gpu_opt_on,draft_elapsed_s,0.023866,0.006848,0.042357,True,12,20000
1,gpu_opt_off,gpu_opt_on,verify_elapsed_s,0.010596,0.003496,0.019330,True,12,20000
2,gpu_opt_off,gpu_opt_on,latency_s,0.035458,0.011016,0.062692,True,12,20000
3,gpu_opt_off,gpu_opt_on,ttft_ms,5.763333,1.994146,12.266042,True,12,20000


In [11]:
runs_csv = results_dir / 'gpu_opt_benchmark_runs.csv'
delta_csv = results_dir / 'gpu_opt_benchmark_delta.csv'
task_delta_csv = results_dir / 'gpu_opt_benchmark_task_delta.csv'
equiv_csv = results_dir / 'gpu_opt_benchmark_equivalence.csv'
robust_csv = results_dir / 'gpu_opt_benchmark_robust_delta.csv'
bootstrap_ci_csv = results_dir / 'gpu_opt_benchmark_bootstrap_ci.csv'

# Backward-compatible task delta summary (for OFF vs ON when present).
task_delta_rows = []
if not paired_df.empty:
    for cond in sorted(paired_df['compare_condition'].unique()):
        for task, g in paired_df[paired_df['compare_condition'] == cond].groupby('task'):
            row = {
                'baseline_condition': baseline_condition,
                'compare_condition': cond,
                'task': task,
            }
            for metric in ['draft_elapsed_s', 'verify_elapsed_s', 'latency_s']:
                base_mean = float(g[f'{metric}_base'].mean())
                cmp_mean = float(g[f'{metric}_cmp'].mean())
                row[f'{metric}_base'] = base_mean
                row[f'{metric}_compare'] = cmp_mean
                row[f'{metric}_delta_pct'] = ((cmp_mean - base_mean) / base_mean * 100.0) if base_mean != 0 else np.nan
            task_delta_rows.append(row)
task_delta_table = pd.DataFrame(task_delta_rows)

df_runs.to_csv(runs_csv, index=False)
mean_delta_table.to_csv(delta_csv, index=False)
task_delta_table.to_csv(task_delta_csv, index=False)
equivalence_table.to_csv(equiv_csv, index=False)
robust_delta_table.to_csv(robust_csv, index=False)
bootstrap_ci_table.to_csv(bootstrap_ci_csv, index=False)

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
df_runs.to_csv(results_dir / f'gpu_opt_benchmark_runs_{stamp}.csv', index=False)
mean_delta_table.to_csv(results_dir / f'gpu_opt_benchmark_delta_{stamp}.csv', index=False)
bootstrap_ci_table.to_csv(results_dir / f'gpu_opt_benchmark_bootstrap_ci_{stamp}.csv', index=False)

print('Saved benchmark outputs:')
print(f'  {runs_csv}')
print(f'  {delta_csv}')
print(f'  {task_delta_csv}')
print(f'  {equiv_csv}')
print(f'  {robust_csv}')
print(f'  {bootstrap_ci_csv}')

mean_delta_table

Saved benchmark outputs:
  C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\gpu_opt_benchmark_runs.csv
  C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\gpu_opt_benchmark_delta.csv
  C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\gpu_opt_benchmark_task_delta.csv
  C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\gpu_opt_benchmark_equivalence.csv
  C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\gpu_opt_benchmark_robust_delta.csv
  C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\gpu_opt_benchmark_bootstrap_ci.csv


,baseline_condition,compare_condition,metric,baseline_mean,compare_mean,abs_delta_compare_minus_base,pct_delta_compare_minus_base,improvement_pct_lower_is_better,ratio_compare_over_base,n_pairs
0,gpu_opt_off,gpu_opt_on,draft_elapsed_s,1.865171,1.889037,0.023866,1.279588,-1.279588,1.012796,12
1,gpu_opt_off,gpu_opt_on,verify_elapsed_s,0.402933,0.413529,0.010596,2.629613,-2.629613,1.026296,12
2,gpu_opt_off,gpu_opt_on,latency_s,2.272600,2.308058,0.035458,1.560254,-1.560254,1.015603,12
3,gpu_opt_off,gpu_opt_on,ttft_ms,170.904167,176.667500,5.763333,3.372260,-3.372260,1.033723,12
